# Anomaly Detection Pipeline — Iter 3

## What went wrong in Iter 2 (OOF F1=0.68 → Codabench F1=0.056)?

The catastrophic train/test gap was caused by **seven compounding failures**:

| # | Root Cause | Fix in Iter 3 |
|---|---|---|
| 1 | RF/ET/GB with 400 trees memorize 1100-user training set | LightGBM (heavy reg) + RF (shallow) + LogReg |
| 2 | Random oversampling duplicates OOF signal, inflating scores | SMOTE applied inside each fold **after** scaling |
| 3 | Ensemble weights optimised for AUC, not F1 | `neg_f1_blend()` searches threshold jointly |
| 4 | 15 SVD components capture training noise | SVD→5 components; use **reconstruction error** only |
| 5 | IsolationForest + LOF imported but never used | Fitted on training, scores added as meta-features |
| 6 | Threshold chosen on the OOF set used for weight optimisation | Median of per-fold best thresholds (nested CV) |
| 7 | No rating-count signal | z-score + percentile rank of `num_ratings`; per-item z-score mean |

## Section 1 — Imports & Config

Added: `lightgbm` for the primary regularised boosting model and `imblearn.over_sampling.SMOTE` to replace naive random duplication. Everything from iter2 is retained.

In [1]:
# Install extra dependencies if running on Colab (already present in most envs)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "imbalanced-learn"], check=True)
print("Dependencies ready.")

Dependencies ready.


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, skew, kurtosis, percentileofscore
from scipy.sparse import csr_matrix
from scipy.optimize import minimize

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score
)

import lightgbm as lgb
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

# ── Configuration ────────────────────────────────────────────────────────────
RANDOM_STATE   = 42
N_ITEMS        = 1000
N_FOLDS        = 5
SVD_COMPONENTS = 5      # reduced from 15: fewer latent dims → less overfitting

print("All imports successful.")

All imports successful.


In [3]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
# ── ⚠️  Update this path to where your .npz files live ──────────────────────
os.chdir("/content/drive/My Drive/cs421 principles of machine learning/group project/leong/iter 3")
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/My Drive/cs421 principles of machine learning/group project/leong/iter 3


## Section 2 — Data Loading & EDA

In [4]:
# ── Load Training Data ────────────────────────────────────────────────────────
data_train = np.load("training_batch_with_labels.npz")
X_raw      = data_train["X"]   # (interactions, 3): [user, item, rating]
y_raw      = data_train["y"]   # (users, 2):        [user, label]

ratings = pd.DataFrame(X_raw, columns=["user", "item", "rating"])
labels  = pd.DataFrame(y_raw, columns=["user", "label"]).set_index("user")

print("=== Training Data ===")
print(f"Total interactions : {len(ratings):,}")
print(f"Unique users       : {ratings['user'].nunique()}")
print(f"Unique items       : {ratings['item'].nunique()}")
print(f"Rating range       : {ratings['rating'].min()} – {ratings['rating'].max()}")
print(f"\nLabel distribution:")
print(labels['label'].value_counts().rename({0: 'Normal', 1: 'Anomalous'}))

imbalance_ratio = (labels['label']==0).sum() / (labels['label']==1).sum()
print(f"\nImbalance ratio (normal:anomalous) = {imbalance_ratio:.1f}:1")

=== Training Data ===
Total interactions : 177,346
Unique users       : 1100
Unique items       : 993
Rating range       : 0 – 5

Label distribution:
label
Normal       1000
Anomalous     100
Name: count, dtype: int64

Imbalance ratio (normal:anomalous) = 10.0:1


## Section 3 — Feature Engineering

**Key changes vs iter2:**
- SVD reduced to 5 components; **raw SVD columns are dropped** — only `svd_reconstruction_error` is kept.
  *Why: 15 SVD components over 1100 users captured training-specific noise. Reconstruction error is a scalar unsupervised anomaly signal that generalises to OOD users.*
- `rating_count_zscore` + `rating_count_percentile`: anomalous users often sit at the extremes of the interaction-count distribution.
- `per_item_zscore_mean`: for each user, how far (on average) are their ratings from what everyone else gave each item? Systematically high values are a strong shill/bot indicator.

In [5]:
def build_features(
    df: pd.DataFrame,
    global_item_means=None,
    svd_model=None,
    global_item_stds=None,
    count_mean: float = None,
    count_std: float  = None,
    all_counts=None,          # needed for percentile calculation on test
    n_items: int = N_ITEMS,
    fit_svd: bool = False,
):
    """
    Build rich user-level features from raw [user, item, rating] interactions.

    Parameters
    ----------
    df               : DataFrame with columns [user, item, rating]
    global_item_means: item mean ratings from training (None → compute here)
    svd_model        : pre-fitted TruncatedSVD (None → fit here if fit_svd=True)
    global_item_stds : item std ratings from training (None → compute here)
    count_mean/std   : training-set mean/std of num_ratings (for z-score on test)
    all_counts       : training-set num_ratings array (for percentile on test)
    fit_svd          : True → fit SVD; False → transform only

    Returns
    -------
    features, global_item_means, svd_model, global_item_stds, count_mean, count_std, all_counts
    """

    # ── Helper functions ──────────────────────────────────────────────────────
    def safe_skew(x):
        return float(skew(x)) if len(x) >= 3 else 0.0

    def safe_kurt(x):
        return float(kurtosis(x)) if len(x) >= 4 else 0.0

    def rating_entropy(x):
        """Shannon entropy over the 0–5 rating distribution."""
        counts = np.bincount(x.astype(int), minlength=6)
        p = counts / counts.sum()
        return float(entropy(p + 1e-9))

    def extreme_prop(x):
        return float(((x == 0) | (x == 5)).mean())

    def high_prop(x):
        return float((x >= 4).mean())

    def low_prop(x):
        return float((x <= 1).mean())

    def repeated_rating_prop(x):
        if len(x) < 2:
            return 0.0
        arr = x.values
        return float((arr[1:] == arr[:-1]).mean())

    def consec_diff_std(x):
        if len(x) < 2:
            return 0.0
        return float(np.std(np.diff(x.values)))

    # ── Group 1: Basic statistics ─────────────────────────────────────────────
    basic = df.groupby("user")["rating"].agg(
        mean_rating   = "mean",
        std_rating    = "std",
        num_ratings   = "count",
        min_rating    = "min",
        max_rating    = "max",
        median_rating = "median",
    ).fillna(0)
    basic["rating_range"] = basic["max_rating"] - basic["min_rating"]

    # ── Group 2: Higher moments ───────────────────────────────────────────────
    moments = df.groupby("user")["rating"].agg(
        skewness = safe_skew,
        kurt     = safe_kurt,
    )

    # ── Group 3: Rating distribution ──────────────────────────────────────────
    dist = df.groupby("user")["rating"].agg(
        rating_entropy      = rating_entropy,
        extreme_rating_prop = extreme_prop,
        high_rating_prop    = high_prop,
        low_rating_prop     = low_prop,
    )

    # ── Group 4: Interaction diversity ────────────────────────────────────────
    diversity = df.groupby("user").agg(num_unique_items=("item", "nunique"))
    diversity["item_diversity_ratio"] = (
        diversity["num_unique_items"] / basic["num_ratings"]
    )

    # ── Group 5: Deviation from global item means ─────────────────────────────
    if global_item_means is None:
        global_item_means = df.groupby("item")["rating"].mean()

    df2 = df.copy()
    df2["item_mean"] = (
        df2["item"].map(global_item_means).fillna(df["rating"].mean())
    )
    df2["deviation"] = df2["rating"] - df2["item_mean"]

    dev_feats = df2.groupby("user")["deviation"].agg(
        mean_deviation     = "mean",
        std_deviation      = "std",
        abs_mean_deviation = lambda x: x.abs().mean(),
    ).fillna(0)

    # ── Group 6: Item popularity ───────────────────────────────────────────────
    item_popularity = df.groupby("item")["rating"].count()
    df3 = df.copy()
    df3["item_pop"] = df3["item"].map(item_popularity).fillna(1)

    pop_feats = df3.groupby("user").agg(
        mean_item_popularity = ("item_pop", "mean"),
        std_item_popularity  = ("item_pop", "std"),
    ).fillna(0)

    # ── Group 7: Sequential / burstiness ──────────────────────────────────────
    bursty = df.groupby("user")["rating"].agg(
        repeated_rating_prop = repeated_rating_prop,
        consec_diff_std      = consec_diff_std,
    )

    # ── Group 8: SVD reconstruction error (NEW — replaces raw SVD components) ─
    # Raw SVD components on 1100 users capture training noise.
    # Reconstruction error is a single scalar: how poorly the low-rank model
    # explains a user's ratings. Anomalous users tend to have high error because
    # their rating pattern doesn't conform to any latent factor structure.
    all_users = sorted(df["user"].unique())
    user_idx  = {u: i for i, u in enumerate(all_users)}

    rows = df["user"].map(user_idx).values
    cols = np.clip(df["item"].values.astype(int), 0, n_items - 1)
    vals = df["rating"].values.astype(float)

    mat = csr_matrix((vals, (rows, cols)), shape=(len(all_users), n_items))
    mat_dense = mat.toarray()  # needed for reconstruction error

    if fit_svd:
        svd_model = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)
        svd_feats = svd_model.fit_transform(mat)
    else:
        svd_feats = svd_model.transform(mat)

    # Reconstruct the matrix and compute per-user MSE
    reconstructed = svd_feats @ svd_model.components_  # shape: (n_users, n_items)
    # Mask: only compute error on items the user actually rated (non-zero entries)
    rated_mask = (mat_dense != 0).astype(float)
    sq_err = (mat_dense - reconstructed) ** 2 * rated_mask
    rated_counts = rated_mask.sum(axis=1).clip(min=1)  # avoid div-by-zero
    reconstruction_error = sq_err.sum(axis=1) / rated_counts  # per-user MSE

    svd_df = pd.DataFrame(
        {"svd_reconstruction_error": reconstruction_error},
        index=all_users,
    )
    # NOTE: raw svd_0...svd_k columns are intentionally NOT added here

    # ── Group 9: Rating-count normalisation (NEW) ─────────────────────────────
    # Anomalous users (bots/shills) often have abnormally high or low interaction
    # counts relative to the user population.
    counts_series = basic["num_ratings"]  # indexed by user

    if count_mean is None:
        # Training: compute population statistics
        count_mean = float(counts_series.mean())
        count_std  = float(counts_series.std()) + 1e-9
        all_counts = counts_series.values.copy()  # keep for percentile on test

    count_norm = pd.DataFrame(index=counts_series.index)
    count_norm["rating_count_zscore"] = (
        (counts_series - count_mean) / count_std
    )
    # Percentile rank: how high (or low) is this user's count vs the population?
    count_norm["rating_count_percentile"] = counts_series.apply(
        lambda c: percentileofscore(all_counts, c, kind="rank") / 100.0
    )

    # ── Group 10: Per-item z-score mean (NEW) ─────────────────────────────────
    # For each (item, rating), compute how many stds above/below the item mean
    # this rating is. Then take each user's mean of these z-scores.
    # Anomalous users systematically give ratings that are statistical outliers
    # per item — captured as a high absolute mean z-score.
    if global_item_stds is None:
        global_item_stds = df.groupby("item")["rating"].std().fillna(1.0)

    df4 = df.copy()
    df4["item_mean"] = df4["item"].map(global_item_means).fillna(df["rating"].mean())
    df4["item_std"]  = df4["item"].map(global_item_stds).fillna(1.0).clip(lower=0.1)
    df4["item_zscore"] = (df4["rating"] - df4["item_mean"]) / df4["item_std"]

    zscore_feats = df4.groupby("user")["item_zscore"].agg(
        per_item_zscore_mean    = "mean",
        per_item_zscore_abs_mean= lambda x: x.abs().mean(),
    )

    # ── Combine all feature groups ────────────────────────────────────────────
    features = (
        basic
        .join(moments)
        .join(dist)
        .join(diversity)
        .join(dev_feats)
        .join(pop_feats)
        .join(bursty)
        .join(svd_df)        # reconstruction error only (no raw SVD cols)
        .join(count_norm)
        .join(zscore_feats)
    ).fillna(0)

    return features, global_item_means, svd_model, global_item_stds, count_mean, count_std, all_counts


# ── Build training features ───────────────────────────────────────────────────
print("Building training features...")
(
    X_feat,
    global_item_means,
    svd_model,
    global_item_stds,
    count_mean,
    count_std,
    all_counts,
) = build_features(ratings, fit_svd=True)

y_series = labels.loc[X_feat.index]["label"]

print(f"Feature matrix shape : {X_feat.shape}")
print(f"Features: {list(X_feat.columns)}")
print(f"\nFeature groups:")
print(f"  Basic stats (7)             : mean, std, count, min, max, median, range")
print(f"  Higher moments (2)          : skewness, kurtosis")
print(f"  Distribution (4)            : entropy, extreme/high/low props")
print(f"  Diversity (2)               : unique items, diversity ratio")
print(f"  Item deviation (3)          : mean, std, abs deviation from item means")
print(f"  Popularity (2)              : mean/std of item popularity")
print(f"  Sequential (2)              : repeated rating prop, consec diff std")
print(f"  SVD reconstruction (1)      : svd_reconstruction_error [NEW, replaces 15 raw SVD cols]")
print(f"  Rating-count norm (2)       : z-score + percentile rank [NEW]")
print(f"  Per-item z-score (2)        : mean + abs-mean of item-z-scores [NEW]")
print(f"  TOTAL: {X_feat.shape[1]} features (vs 37 in iter2)")

Building training features...
Feature matrix shape : (1100, 27)
Features: ['mean_rating', 'std_rating', 'num_ratings', 'min_rating', 'max_rating', 'median_rating', 'rating_range', 'skewness', 'kurt', 'rating_entropy', 'extreme_rating_prop', 'high_rating_prop', 'low_rating_prop', 'num_unique_items', 'item_diversity_ratio', 'mean_deviation', 'std_deviation', 'abs_mean_deviation', 'mean_item_popularity', 'std_item_popularity', 'repeated_rating_prop', 'consec_diff_std', 'svd_reconstruction_error', 'rating_count_zscore', 'rating_count_percentile', 'per_item_zscore_mean', 'per_item_zscore_abs_mean']

Feature groups:
  Basic stats (7)             : mean, std, count, min, max, median, range
  Higher moments (2)          : skewness, kurtosis
  Distribution (4)            : entropy, extreme/high/low props
  Diversity (2)               : unique items, diversity ratio
  Item deviation (3)          : mean, std, abs deviation from item means
  Popularity (2)              : mean/std of item popularit

## Section 4 — Unsupervised Meta-Features (IsolationForest + LOF)

**Why this matters:** Supervised models trained on 1100 users struggle to generalise to OOD test users. Unsupervised anomaly detectors are label-free — they identify structural outliers in feature space, which remains informative even for users never seen during training.

**Critical rule:** Both models are fitted **on training data only** and applied to test data via `transform()`/`predict()`. Re-fitting on test would leak test distribution information.

In [6]:
X_arr = X_feat.values
y_arr = y_series.values

# Scale first — unsupervised models need comparable feature ranges
# We use a single global scaler here for the unsupervised step only.
# Each CV fold will have its own scaler for supervised training.
global_scaler = StandardScaler()
X_arr_s = global_scaler.fit_transform(X_arr)

# ── IsolationForest ──────────────────────────────────────────────────────────
# contamination=0.09 ≈ our 10:1 imbalance (9% anomalous).
# score_samples() returns a score where MORE NEGATIVE = MORE ANOMALOUS.
# We negate so that higher score = more anomalous (consistent with other signals).
print("Fitting IsolationForest...")
iso = IsolationForest(
    n_estimators=200,
    contamination=0.09,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_arr_s)
iso_scores_train = -iso.score_samples(X_arr_s)  # negate: higher = more anomalous

# ── Local Outlier Factor (novelty=True for test-time transform) ───────────────
# novelty=True is mandatory: without it, LOF cannot score unseen test points.
# negative_outlier_factor_ on training is already available after fit.
print("Fitting LocalOutlierFactor...")
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.09,
    novelty=True,   # MUST be True to call predict/score on test data later
    n_jobs=-1,
)
lof.fit(X_arr_s)
# score_samples gives negative_outlier_factor (less negative = more normal)
# Negate so higher = more anomalous
lof_scores_train = -lof.score_samples(X_arr_s)

# ── Append meta-features to feature array ────────────────────────────────────
# These two columns are added AFTER the global scaler was fitted, so the
# per-fold scaler (inside CV) will also scale them appropriately.
X_feat["iso_score"] = iso_scores_train
X_feat["lof_score"] = lof_scores_train

# Re-read the array with new columns
X_arr = X_feat.values
TRAIN_FEATURE_COLS = list(X_feat.columns)  # save column order for test alignment

print(f"\nAdded iso_score + lof_score → feature matrix now {X_arr.shape}")
print(f"IsoForest anomaly score range : [{iso_scores_train.min():.4f}, {iso_scores_train.max():.4f}]")
print(f"LOF anomaly score range       : [{lof_scores_train.min():.4f}, {lof_scores_train.max():.4f}]")
print(f"\nCorrelation with true labels:")
print(f"  iso_score : {np.corrcoef(iso_scores_train, y_arr)[0,1]:+.4f}")
print(f"  lof_score : {np.corrcoef(lof_scores_train, y_arr)[0,1]:+.4f}")

Fitting IsolationForest...
Fitting LocalOutlierFactor...

Added iso_score + lof_score → feature matrix now (1100, 29)
IsoForest anomaly score range : [0.3653, 0.6974]
LOF anomaly score range       : [0.9489, 4.0472]

Correlation with true labels:
  iso_score : +0.2259
  lof_score : -0.0447


## Section 5 — CV Training Loop (LightGBM + RF + LogReg, SMOTE per fold)

**Model choices:**
- **LightGBM (weight 0.5):** Gradient boosting with aggressive L1/L2 regularisation (`reg_alpha=1`, `reg_lambda=5`), shallow trees (`num_leaves=15`, `max_depth=4`), and `min_child_samples=20`. These hyperparameters specifically prevent memorisation of the small training set.
- **RandomForest (weight 0.3):** Shallow (`max_depth=7`), min leaf size 5. Provides diversity from LightGBM's sequential boosting.
- **LogisticRegression (weight 0.2):** Heavy L2 penalty (`C=0.1`). Acts as a high-bias / low-variance anchor that pulls the ensemble away from overfit regions.

**SMOTE inside each fold:** Synthetic minority oversampling creates interpolated anomalous users between existing ones, rather than just duplicating them. Applied AFTER per-fold scaling, NEVER on validation folds.

In [7]:
# ── Model definitions ─────────────────────────────────────────────────────────
MODEL_CONFIGS = {
    "LightGBM": dict(
        estimator = lgb.LGBMClassifier,
        params = dict(
            n_estimators      = 500,
            num_leaves        = 15,       # small: prevent memorisation
            max_depth         = 4,        # shallow trees generalise better
            learning_rate     = 0.03,     # slow learning with many weak trees
            min_child_samples = 20,       # each leaf needs ≥20 samples
            reg_alpha         = 1.0,      # L1 regularisation
            reg_lambda        = 5.0,      # L2 regularisation (strong)
            scale_pos_weight  = 10,       # compensates 10:1 class imbalance
            subsample         = 0.7,      # row subsampling per tree
            colsample_bytree  = 0.7,      # column subsampling per tree
            random_state      = RANDOM_STATE,
            n_jobs            = -1,
            verbose           = -1,       # suppress LightGBM output
        ),
    ),
    "RandomForest": dict(
        estimator = RandomForestClassifier,
        params = dict(
            n_estimators     = 300,
            max_depth        = 7,         # shallower than iter2's depth-10
            min_samples_leaf = 5,         # bigger than iter2's min=3
            class_weight     = "balanced",
            max_features     = "sqrt",
            random_state     = RANDOM_STATE,
            n_jobs           = -1,
        ),
    ),
    "LogisticRegression": dict(
        estimator = LogisticRegression,
        params = dict(
            C            = 0.1,           # strong regularisation (C<1 = more reg)
            class_weight = "balanced",
            solver       = "lbfgs",
            max_iter     = 1000,
            random_state = RANDOM_STATE,
        ),
    ),
}

model_names = list(MODEL_CONFIGS.keys())
oof_preds   = {name: np.zeros(len(y_arr)) for name in model_names}

# ── Cross-Validation Loop ─────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_scalers       = []                         # per-fold scalers for test inference
fold_models        = {name: [] for name in model_names}
fold_best_thresholds = []                       # for nested threshold CV (Section 7)

print(f"Starting {N_FOLDS}-Fold Stratified CV...")
print(f"Models: {model_names}\n")

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_arr, y_arr)):
    X_tr, X_va = X_arr[train_idx], X_arr[val_idx]
    y_tr, y_va = y_arr[train_idx], y_arr[val_idx]

    # ── Scale BEFORE SMOTE (fit on training fold only) ────────────────────────
    # Scaling before SMOTE ensures synthetic points are interpolated in the
    # normalised space — more geometrically meaningful.
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)   # fit on train fold only
    X_va_s   = scaler.transform(X_va)        # transform val with same scaler
    fold_scalers.append(scaler)

    # ── SMOTE on training fold only ───────────────────────────────────────────
    # k_neighbors=3 because the minority class has only ~80 samples in each fold
    # (100 anomalous × 4/5 ≈ 80). With k_neighbors=5 (default) we risk
    # synthetic samples that span too wide a region.
    n_minority = (y_tr == 1).sum()
    k_nn = min(3, n_minority - 1)   # guard: k < n_minority
    smote = SMOTE(k_neighbors=k_nn, random_state=RANDOM_STATE + fold_idx)
    X_tr_aug, y_tr_aug = smote.fit_resample(X_tr_s, y_tr)
    # Validation fold is NEVER augmented

    fold_aucs = {}
    fold_oof_blend = np.zeros(len(val_idx))  # for fold-level threshold search

    for name, cfg in MODEL_CONFIGS.items():
        clf = cfg["estimator"](**cfg["params"])
        clf.fit(X_tr_aug, y_tr_aug)
        oof_preds[name][val_idx] = clf.predict_proba(X_va_s)[:, 1]
        fold_aucs[name] = roc_auc_score(y_va, oof_preds[name][val_idx])
        fold_models[name].append(clf)

    # Collect per-fold best threshold (using preliminary equal weights) for
    # cross-validated threshold estimation in Section 7
    # (equal weights here; optimised weights computed later in Section 6)
    fold_blend = np.mean(
        [oof_preds[n][val_idx] for n in model_names], axis=0
    )
    best_t, best_f1_t = 0.5, 0.0
    for t in np.linspace(0.05, 0.95, 200):
        f1t = f1_score(y_va, (fold_blend >= t).astype(int), zero_division=0)
        if f1t > best_f1_t:
            best_f1_t, best_t = f1t, t
    fold_best_thresholds.append(best_t)

    avg_fold_auc = np.mean(list(fold_aucs.values()))
    print(
        f"Fold {fold_idx+1}/{N_FOLDS}  "
        + "  ".join(f"{n}: {a:.4f}" for n, a in fold_aucs.items())
        + f"  │  Avg: {avg_fold_auc:.4f}  │  fold_thresh: {best_t:.3f}"
    )

print("\n" + "="*70)
print("OVERALL OOF AUC (full dataset):")
for name, oof in oof_preds.items():
    print(f"  {name:<22} AUC = {roc_auc_score(y_arr, oof):.4f}")

Starting 5-Fold Stratified CV...
Models: ['LightGBM', 'RandomForest', 'LogisticRegression']

Fold 1/5  LightGBM: 0.9080  RandomForest: 0.8710  LogisticRegression: 0.8910  │  Avg: 0.8900  │  fold_thresh: 0.642
Fold 2/5  LightGBM: 0.9142  RandomForest: 0.9427  LogisticRegression: 0.8978  │  Avg: 0.9182  │  fold_thresh: 0.588
Fold 3/5  LightGBM: 0.9600  RandomForest: 0.9358  LogisticRegression: 0.9670  │  Avg: 0.9543  │  fold_thresh: 0.719
Fold 4/5  LightGBM: 0.9193  RandomForest: 0.9183  LogisticRegression: 0.9080  │  Avg: 0.9152  │  fold_thresh: 0.787
Fold 5/5  LightGBM: 0.9293  RandomForest: 0.9083  LogisticRegression: 0.8885  │  Avg: 0.9087  │  fold_thresh: 0.448

OVERALL OOF AUC (full dataset):
  LightGBM               AUC = 0.9266
  RandomForest           AUC = 0.9147
  LogisticRegression     AUC = 0.9106


## Section 6 — Ensemble Weight Optimisation (F1-targeted)

**The iter2 bug:** Nelder-Mead minimised `-AUC`. AUC measures rank ordering, not whether the predictions cross a good threshold. A model that perfectly separates classes in rank order but pushes all scores to 0.99 would score perfectly on AUC but fail F1.

**The fix:** `neg_f1_blend()` jointly searches over both weights AND threshold, returning the best F1 achievable. This directly optimises the metric we care about.

In [8]:
oof_list = [oof_preds[name] for name in model_names]
n_models = len(oof_list)

def neg_f1_blend(weights, oof_list, y):
    """
    Objective for Nelder-Mead: minimise negative F1 of the weighted blend.
    Jointly searches over weights AND threshold — avoids the AUC/F1 misalignment.
    """
    w = np.maximum(weights, 0)       # enforce non-negative weights
    w = w / (w.sum() + 1e-9)         # normalise to sum = 1
    blend = sum(wi * oi for wi, oi in zip(w, oof_list))
    best_f1 = 0.0
    for t in np.linspace(0.05, 0.95, 200):
        f1 = f1_score(y, (blend >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
    return -best_f1  # minimise negative F1

# Start from prescribed initial weights: LightGBM=0.5, RF=0.3, LogReg=0.2
w0 = np.array([0.5, 0.3, 0.2])

result = minimize(
    neg_f1_blend,
    w0,
    args=(oof_list, y_arr),
    method="Nelder-Mead",
    options={"maxiter": 10_000, "xatol": 1e-7, "fatol": 1e-7},
)

opt_weights = np.maximum(result.x, 0)
opt_weights = opt_weights / opt_weights.sum()

print("Optimised ensemble weights (F1-targeted):")
for name, w in zip(model_names, opt_weights):
    print(f"  {name:<22} weight = {w:.4f}")

# Blended OOF prediction
blended_oof = sum(w * o for w, o in zip(opt_weights, oof_list))
blended_auc = roc_auc_score(y_arr, blended_oof)

print(f"\nBlended OOF AUC : {blended_auc:.4f}")

Optimised ensemble weights (F1-targeted):
  LightGBM               weight = 0.4926
  RandomForest           weight = 0.3103
  LogisticRegression     weight = 0.1970

Blended OOF AUC : 0.9342


## Section 7 — Cross-Validated Threshold Tuning

**The iter2 bug:** The threshold (0.553) was chosen on the same OOF predictions used to optimise the ensemble weights — circular evaluation that biases the threshold upward.

**The fix:** Per-fold best thresholds were collected during the CV loop (Section 5). We use the **median** of these thresholds as the final threshold. The standard deviation tells us how stable the threshold is across folds — high std means the threshold is data-dependent and we should trust it less.

We still save **continuous scores** as the final submission (not binary), since Codabench computes AUC from the score.

In [9]:
# Per-fold thresholds were collected in Section 5
thresh_array  = np.array(fold_best_thresholds)
final_thresh  = float(np.median(thresh_array))  # median is robust to outlier folds
thresh_std    = float(np.std(thresh_array))

print("Cross-validated threshold analysis:")
print(f"  Per-fold best thresholds : {thresh_array.round(3).tolist()}")
print(f"  Median threshold         : {final_thresh:.4f}")
print(f"  Std of thresholds        : {thresh_std:.4f}")
if thresh_std > 0.1:
    print("  ⚠️  High std → threshold is unstable; continuous scores are safer.")
else:
    print("  ✓  Low std → threshold is stable across folds.")

# Also search for best threshold on blended OOF (for reporting only)
best_thresh_oof, best_f1_oof = 0.5, 0.0
for t in np.linspace(0.05, 0.95, 200):
    f1t = f1_score(y_arr, (blended_oof >= t).astype(int), zero_division=0)
    if f1t > best_f1_oof:
        best_f1_oof, best_thresh_oof = f1t, t

print(f"\nBest OOF threshold (for reference, NOT used for submission) : {best_thresh_oof:.4f}")
print(f"Best OOF F1 at that threshold                              : {best_f1_oof:.4f}")

Cross-validated threshold analysis:
  Per-fold best thresholds : [0.642, 0.588, 0.719, 0.787, 0.448]
  Median threshold         : 0.6425
  Std of thresholds        : 0.1162
  ⚠️  High std → threshold is unstable; continuous scores are safer.

Best OOF threshold (for reference, NOT used for submission) : 0.7193
Best OOF F1 at that threshold                              : 0.6893


## Section 8 — Full Evaluation Report

In [10]:
# Use the cross-validated threshold for binary classification
y_pred_cv    = (blended_oof >= final_thresh).astype(int)
y_pred_oof   = (blended_oof >= best_thresh_oof).astype(int)  # oracle OOF threshold

print("=" * 60)
print("FULL EVALUATION REPORT — Iter 3 OOF predictions")
print("=" * 60)

print(f"\n── Using cross-validated threshold ({final_thresh:.3f}) ──")
print(f"  AUC       : {roc_auc_score(y_arr, blended_oof):.4f}")
print(f"  Precision : {precision_score(y_arr, y_pred_cv):.4f}")
print(f"  Recall    : {recall_score(y_arr, y_pred_cv):.4f}")
print(f"  F1 Score  : {f1_score(y_arr, y_pred_cv):.4f}")

print(f"\n── Using OOF-oracle threshold ({best_thresh_oof:.3f}) [upper bound] ──")
print(f"  Precision : {precision_score(y_arr, y_pred_oof):.4f}")
print(f"  Recall    : {recall_score(y_arr, y_pred_oof):.4f}")
print(f"  F1 Score  : {f1_score(y_arr, y_pred_oof):.4f}")

print("\n--- Baseline Comparison ---")
print(f"{'Metric':<15} {'Iter1(base)':>12} {'Iter2(OOF)':>12} {'Iter3(OOF)':>12}")
print("-" * 55)
iter3_auc  = roc_auc_score(y_arr, blended_oof)
iter3_prec = precision_score(y_arr, y_pred_cv)
iter3_rec  = recall_score(y_arr, y_pred_cv)
iter3_f1   = f1_score(y_arr, y_pred_cv)
for metric, b1, b2, b3 in [
    ("AUC",       0.8573, 0.9448, iter3_auc),
    ("Precision", 0.7500, 0.8750, iter3_prec),
    ("Recall",    0.4500, 0.5600, iter3_rec),
    ("F1",        0.5625, 0.6829, iter3_f1),
]:
    print(f"{metric:<15} {b1:>12.4f} {b2:>12.4f} {b3:>12.4f}")

print("\n--- Per-Model OOF AUC ---")
for name, oof in oof_preds.items():
    print(f"  {name:<22} {roc_auc_score(y_arr, oof):.4f}")

# ── Anti-overfitting sanity check ─────────────────────────────────────────────
# A significant train-OOF gap would still signal memorisation
print("\n--- Overfitting Sanity Check ---")
for name, oof in oof_preds.items():
    # Refit on ALL training data and score it (in-sample AUC)
    clf_full = MODEL_CONFIGS[name]["estimator"](**MODEL_CONFIGS[name]["params"])
    sc_full  = StandardScaler().fit(X_arr)
    clf_full.fit(sc_full.transform(X_arr), y_arr)
    train_auc = roc_auc_score(y_arr, clf_full.predict_proba(sc_full.transform(X_arr))[:, 1])
    oof_auc   = roc_auc_score(y_arr, oof)
    gap       = train_auc - oof_auc
    flag      = "⚠️  OVERFIT" if gap > 0.1 else "✓"
    print(f"  {name:<22} Train AUC={train_auc:.4f}  OOF AUC={oof_auc:.4f}  Gap={gap:+.4f}  {flag}")

FULL EVALUATION REPORT — Iter 3 OOF predictions

── Using cross-validated threshold (0.642) ──
  AUC       : 0.9342
  Precision : 0.6737
  Recall    : 0.6400
  F1 Score  : 0.6564

── Using OOF-oracle threshold (0.719) [upper bound] ──
  Precision : 0.7922
  Recall    : 0.6100
  F1 Score  : 0.6893

--- Baseline Comparison ---
Metric           Iter1(base)   Iter2(OOF)   Iter3(OOF)
-------------------------------------------------------
AUC                   0.8573       0.9448       0.9342
Precision             0.7500       0.8750       0.6737
Recall                0.4500       0.5600       0.6400
F1                    0.5625       0.6829       0.6564

--- Per-Model OOF AUC ---
  LightGBM               0.9266
  RandomForest           0.9147
  LogisticRegression     0.9106

--- Overfitting Sanity Check ---
  LightGBM               Train AUC=1.0000  OOF AUC=0.9266  Gap=+0.0734  ✓
  RandomForest           Train AUC=0.9977  OOF AUC=0.9147  Gap=+0.0830  ✓
  LogisticRegression     Train AUC=0.

## Section 9 — Test Prediction & Submission

**Key changes vs iter2:**
1. Test file is `second_batch.npz` (key: `X`).
2. IsolationForest and LOF scores are computed using the **training-fitted models** (no refitting).
3. Feature column order is enforced to match training exactly via `TRAIN_FEATURE_COLS`.
4. Submission saved as `submission_batch2.npz`.

In [11]:
# ── Load test data ────────────────────────────────────────────────────────────
data_test  = np.load("second_batch.npz")
X_test_raw = data_test["X"]
df_test    = pd.DataFrame(X_test_raw, columns=["user", "item", "rating"])

print(f"Test interactions  : {len(df_test):,}")
print(f"Test unique users  : {df_test['user'].nunique()}")

# ── Build test features using training-fitted statistics ──────────────────────
print("\nBuilding test features (using training statistics)...")
(
    X_test_feat,
    _,  # global_item_means already fitted
    _,  # svd_model already fitted
    _,  # global_item_stds already fitted
    _,  # count_mean already fitted
    _,  # count_std already fitted
    _,  # all_counts already fitted
) = build_features(
    df_test,
    global_item_means = global_item_means,  # from training
    svd_model         = svd_model,          # from training
    global_item_stds  = global_item_stds,   # from training
    count_mean        = count_mean,         # from training
    count_std         = count_std,          # from training
    all_counts        = all_counts,         # from training (for percentile)
    fit_svd           = False,              # MUST be False for test
)

print(f"Test feature matrix (pre meta-features): {X_test_feat.shape}")

# ── Apply training-fitted IsolationForest and LOF ─────────────────────────────
# CRITICAL: We use global_scaler (fitted on training) to scale test data.
# iso and lof were fitted on training and must NOT be refitted.
X_test_arr_raw = X_test_feat.values
X_test_arr_s   = global_scaler.transform(X_test_arr_raw)  # transform only

iso_scores_test = -iso.score_samples(X_test_arr_s)   # negate: higher = more anomalous
lof_scores_test = -lof.score_samples(X_test_arr_s)   # negate: higher = more anomalous

X_test_feat["iso_score"] = iso_scores_test
X_test_feat["lof_score"] = lof_scores_test

# ── Enforce column order matches training exactly ─────────────────────────────
# Any missing column (e.g., items that only appear in test) is filled with 0.
for col in TRAIN_FEATURE_COLS:
    if col not in X_test_feat.columns:
        X_test_feat[col] = 0.0
X_test_feat = X_test_feat[TRAIN_FEATURE_COLS]  # reorder to match training

X_test_arr = X_test_feat.values
test_users  = X_test_feat.index.values
print(f"Test feature matrix (with meta-features): {X_test_arr.shape}")

# ── Fold-averaged predictions per model ───────────────────────────────────────
# Average predictions from all 5 fold-trained models to reduce variance.
# Each fold used its own scaler → apply the matching scaler.
test_preds_per_model = {}

for name in model_names:
    fold_test_scores = []
    for fold_idx, (clf, scaler) in enumerate(zip(fold_models[name], fold_scalers)):
        X_test_s = scaler.transform(X_test_arr)  # use fold-specific scaler
        scores   = clf.predict_proba(X_test_s)[:, 1]
        fold_test_scores.append(scores)
    test_preds_per_model[name] = np.mean(fold_test_scores, axis=0)

# ── Weighted ensemble blend ───────────────────────────────────────────────────
test_oof_list = [test_preds_per_model[name] for name in model_names]
final_scores  = sum(w * o for w, o in zip(opt_weights, test_oof_list))

print(f"\nGenerated {len(final_scores)} anomaly scores for test users.")
print(f"Score range : [{final_scores.min():.4f}, {final_scores.max():.4f}]")
print(f"Mean score  : {final_scores.mean():.4f}")

# Preview top anomalous users
results_df = pd.DataFrame({
    "user"         : test_users,
    "anomaly_score": final_scores,
}).sort_values("anomaly_score", ascending=False)

print("\nTop 10 most anomalous test users:")
print(results_df.head(10).to_string(index=False))

# Optional: binary predictions using cross-validated threshold
n_flagged = (final_scores >= final_thresh).sum()
print(f"\nUsers flagged as anomalous (threshold={final_thresh:.3f}): {n_flagged}")
print(f"Expected ~100 anomalous users; flagging {n_flagged} ({'✓ reasonable' if 80<=n_flagged<=130 else '⚠️ check threshold'})")

Test interactions  : 134,594
Test unique users  : 860

Building test features (using training statistics)...
Test feature matrix (pre meta-features): (860, 27)
Test feature matrix (with meta-features): (860, 29)

Generated 860 anomaly scores for test users.
Score range : [0.0026, 0.9902]
Mean score  : 0.1162

Top 10 most anomalous test users:
 user  anomaly_score
 4139       0.990243
 4537       0.801963
 4809       0.791905
 4485       0.759360
 4811       0.732964
 4339       0.697442
 4567       0.695641
 4331       0.669972
 4845       0.668273
 4862       0.649615

Users flagged as anomalous (threshold=0.642): 11
Expected ~100 anomalous users; flagging 11 (⚠️ check threshold)


In [12]:
# ── Save predictions in required format ───────────────────────────────────────
# Save continuous anomaly scores (not binary). Codabench computes AUC from these.
output_path = "submission_batch2.npz"
np.savez(output_path, predictions=final_scores)

# Verify the saved file
check = np.load(output_path)
assert "predictions" in check, "ERROR: 'predictions' key missing!"
assert len(check["predictions"]) == len(test_users), "ERROR: Length mismatch!"
assert not np.any(np.isnan(check["predictions"])), "ERROR: NaN in predictions!"

print(f"Submission saved to : {output_path}")
print(f"Key                 : 'predictions'")
print(f"Number of scores    : {len(check['predictions'])}")
print(f"Score range         : [{check['predictions'].min():.4f}, {check['predictions'].max():.4f}]")
print("\n✓ Submission verified successfully.")

Submission saved to : submission_batch2.npz
Key                 : 'predictions'
Number of scores    : 860
Score range         : [0.0026, 0.9902]

✓ Submission verified successfully.


## Summary of Changes — Iter 2 → Iter 3

| Component | Iter 2 | Iter 3 | Why |
|---|---|---|---|
| Primary model | GradientBoosting (200 trees) | **LightGBM** (500 trees, reg_λ=5, num_leaves=15) | Aggressive regularisation prevents memorisation of 1100-user set |
| Supporting models | ExtraTrees (removed) | **LogisticRegression** (C=0.1) | High-bias anchor; ExtraTrees correlated with RF, no diversity |
| Oversampling | Random duplication | **SMOTE** (after per-fold scaling) | Synthetic interpolation doesn't duplicate OOF signal |
| Ensemble objective | `-AUC` | **`-F1`** (joint threshold search) | Directly optimises the evaluation metric |
| SVD | 15 raw components | **Reconstruction error only** (5 components) | Scalar unsupervised signal; 15 components capture training noise |
| Unsupervised signals | Imported but unused | **IsoForest + LOF scores** as meta-features | Label-free; generalise to OOD test users |
| Threshold | Chosen on OOF (biased) | **Median of per-fold thresholds** | Avoids circular evaluation |
| New features | — | `rating_count_zscore`, `rating_count_percentile`, `per_item_zscore_mean` | Catch bots with abnormal interaction counts or systematic item-level bias |
| Test file | `first_batch.npz` | `second_batch.npz` | Updated per task specification |